# 02. YOLO 직접 추론 — 라이브러리 없이 전처리 · 출력 해석 · NMS

> edge-lab 서버는 `ultralytics` 라이브러리가 이 과정을 대신한다. 여기서는 **OpenVINO 로 모델 계산만 시키고, 나머지는 전부 직접** 짠다.
> 물체 찾기(object detection) 모델이 실제로 무엇을 내놓고, 그것을 어떻게 "상자 몇 개" 로 줄이는지 손으로 따라간다.

### 이 시간에 배우는 것
1. **letterbox** — 비율을 지키며 정사각형 입력에 넣고, 나중에 되돌리기
2. YOLO 출력 `[1, 84, 8400]` 의 뜻 — 8400 은 어디서 왔고 84 는 무엇인가
3. **IoU** 와 **NMS** — 겹친 상자를 하나로 줄이는 알고리즘을 직접 구현
4. 좌표를 원래 사진으로 되돌리기, 단계별 시간 재기
5. (덤) 포즈 모델 출력 `[1, 56, 8400]` 에서 관절 17개 꺼내기

### 전체 흐름
```
사진 ─letterbox─▶ 640x640 ─RGB·÷255·NCHW─▶ [1,3,640,640]
     ─추론─▶ [1,84,8400] ─전치─▶ 후보 8400개 × (상자 4 + 점수 80)
     ─점수 거르기─▶ 수십 개 ─NMS─▶ 몇 개 ─좌표 되돌리기─▶ 원래 사진 위의 상자
```

## 1. 준비 — 모델과 클래스 이름

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

In [ ]:
YOLO_DIR = MODELS / "object" / "yolo11m_openvino_model"
xml = next(YOLO_DIR.glob("*.xml"))                 # ultralytics 가 내보낸 IR (yolo11m.xml)
model = core.read_model(xml)
print("입력:", model.input(0).any_name, model.input(0).partial_shape)
print("출력:", model.output(0).any_name, model.output(0).partial_shape)

# 클래스 이름 — 내보낼 때 같이 저장된 metadata.yaml 에 있다 (COCO 80종)
import yaml
meta = yaml.safe_load(open(YOLO_DIR / "metadata.yaml", encoding="utf-8"))
NAMES = meta["names"]                              # {0: 'person', 1: 'bicycle', ...}
print(len(NAMES), "종:", list(NAMES.values())[:8], "...")

**YOLO11m** — "한 번 보고(You Only Look Once)" 사진 전체에서 물체 80종을 찾는 모델. `m` 은 중간 크기(약 2천만 개 숫자).
COCO 라는 공개 사진 묶음으로 학습했다. ultralytics 가 OpenVINO 용으로 내보낼 때 입력을 **640 x 640 고정**으로 정했다.

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. letterbox — 비율을 지키며 640 x 640 에 넣기
입력이 정사각형인데 웹캠 사진은 4:3 이다. 그냥 늘리면 사람이 홀쭉해지거나 뚱뚱해져서 모델이 배운 모습과 달라진다.

**방법**: 긴 쪽을 640 에 맞추는 **배율**로 줄이고, 남는 쪽은 회색(114, 114, 114)으로 채운다 — 영화의 위아래 검은 띠(letterbox)와 같다.

예) 480 x 640 사진 → 배율 = min(640/480, 640/640) = **1.0** → 그대로 640 x 480 → 위아래에 (640−480)/2 = **80** 씩 회색.

나중에 상자 좌표를 원래 사진으로 되돌리려면 **배율**과 **여백**을 기억해 둬야 한다.

In [ ]:
SIZE = 640

def letterbox(bgr, size=SIZE):
    h, w = bgr.shape[:2]
    r = min(size / h, size / w)                     # 배율 — 긴 쪽이 size 가 되게
    nw, nh = round(w * r), round(h * r)
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    px, py = (size - nw) / 2, (size - nh) / 2        # 좌우 · 위아래 여백 (가운데 정렬)
    top, bottom = round(py - 0.1), round(py + 0.1)   # 홀수 픽셀일 때 위·아래에 나눠 붙인다
    left, right = round(px - 0.1), round(px + 0.1)
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return img, r, (left, top)

boxed, ratio, pad = letterbox(frame)
print("원래:", frame.shape, "| 배율:", round(ratio, 3), "| 여백(왼쪽, 위):", pad, "→", boxed.shape)
show(boxed, 400)

## 4. 모델 입력으로 — RGB · 0~1 · NCHW
YOLO 는 **RGB** 사진을 **0~1** 로 나눠서 배웠다 (01 번 얼굴 모델은 BGR · 0~255 였다 — 모델마다 다르다!).
`np.ascontiguousarray` 는 축을 바꾼 배열을 메모리에 순서대로 다시 놓는다 (추론 엔진이 더 빨리 읽는다).

In [ ]:
def to_blob(img):
    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)          # YOLO 는 RGB 로 배웠다
    x = x.astype(np.float32) / 255.0                  # 0~255 → 0~1
    x = x.transpose(2, 0, 1)[None]                    # HWC → NCHW
    return np.ascontiguousarray(x)

blob = to_blob(boxed)
print(blob.shape, blob.dtype, "값 범위:", blob.min(), "~", blob.max())

## 5. 추론과 출력 읽기 — `[1, 84, 8400]`

**8400 은 어디서?** YOLO11 은 사진을 세 가지 크기의 격자로 본다. 격자 칸 하나가 "여기에 물체 중심이 있다면" 후보 하나를 낸다.
- 8 픽셀 칸 → 640/8 = 80 × 80 = 6400 칸 (작은 물체)
- 16 픽셀 칸 → 40 × 40 = 1600 칸 (중간)
- 32 픽셀 칸 → 20 × 20 = 400 칸 (큰 물체)
- 합 **8400**

**84 는?** 후보마다 `[중심 x, 중심 y, 너비, 높이]` 4개 + 80종 각각의 점수 80개. 좌표는 640 x 640 입력 기준 픽셀이다.
(옛 YOLOv5 에는 "물체가 있을 확률" 이 따로 있었지만 YOLO11 은 없다 — 80종 점수 중 가장 큰 것이 곧 확신도.)

In [ ]:
print("격자:", {s: (640 // s) ** 2 for s in (8, 16, 32)}, "합:", sum((640 // s) ** 2 for s in (8, 16, 32)))

DEVICE = pick("GPU", "NPU", "CPU")
compiled = core.compile_model(model, DEVICE)
out = compiled(blob)[compiled.output(0)]              # [1, 84, 8400]
print(DEVICE, "출력:", out.shape)

pred = out[0].T                                       # [8400, 84] — 줄 하나가 후보 하나 (전치: 다루기 쉽게)
boxes_cxcywh = pred[:, :4]                            # 가운데 x, 가운데 y, 너비, 높이 (640 좌표)
scores_all = pred[:, 4:]                              # 80종 점수
cls = scores_all.argmax(1)                            # 후보마다 가장 높은 종류
conf = scores_all.max(1)                              # 그 점수
print("점수 0.25 넘는 후보:", int((conf > 0.25).sum()), "/ 8400")

top = np.argsort(-conf)[:5]                           # 점수 높은 다섯
for i in top:
    print(f"  후보 {i:4d}: {NAMES[int(cls[i])]:12s} 점수 {conf[i]:.2f}  중심({boxes_cxcywh[i, 0]:.0f}, {boxes_cxcywh[i, 1]:.0f}) 크기 {boxes_cxcywh[i, 2]:.0f}x{boxes_cxcywh[i, 3]:.0f}")

## 6. IoU — 두 상자가 얼마나 겹치나
같은 사람을 여러 격자 칸이 동시에 찾아서, 거의 같은 상자가 여러 개 나온다. 이것을 하나로 줄이려면 "얼마나 겹치나" 를 숫자로 재야 한다.

$$\text{IoU} = \frac{\text{겹친 넓이}}{\text{두 상자를 합친 넓이}} = \frac{A \cap B}{A + B - A \cap B}$$

0 이면 전혀 안 겹침, 1 이면 똑같은 상자. 손으로 한 번 계산해 보자.

In [ ]:
def iou(a, b):
    """a: [4] 상자 하나, b: [N, 4] 상자들 (x1, y1, x2, y2)"""
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])     # 겹친 부분의 왼쪽 위
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])     # 겹친 부분의 오른쪽 아래
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)      # 안 겹치면 음수 → 0
    area = lambda r: (r[..., 2] - r[..., 0]) * (r[..., 3] - r[..., 1])
    return inter / (area(a) + area(b) - inter + 1e-9)

A = np.array([0, 0, 100, 100.])
B = np.array([[50, 0, 150, 100.],     # 반쯤 겹침: 겹친 넓이 50x100=5000, 합친 넓이 10000+10000-5000=15000 → 0.333
              [0, 0, 100, 100.],      # 똑같음 → 1
              [200, 200, 300, 300.]]) # 안 겹침 → 0
print(np.round(iou(A, B), 3))

## 7. NMS — 겹친 상자 지우기 (Non-Maximum Suppression)
**알고리즘**
1. 점수 높은 순으로 줄 세운다.
2. 맨 앞(가장 높은 것)을 **남긴다**.
3. 그것과 IoU 가 기준(예: 0.45)보다 큰 나머지는 "같은 물체" 로 보고 **버린다**.
4. 남은 것들로 2번부터 되풀이.

먼저 만든 상자로 눈으로 확인해 보자.

In [ ]:
def nms(xyxy, scores, iou_th=0.45):
    order = scores.argsort()[::-1]                    # 점수 높은 순
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)                                # 가장 높은 것을 남기고
        rest = order[1:]
        order = rest[iou(xyxy[i], xyxy[rest]) < iou_th]  # 그것과 많이 겹친 것은 버린다
    return np.array(keep, dtype=int)

toy = np.array([[60, 60, 260, 300], [70, 50, 270, 290], [80, 70, 250, 310],   # 같은 물체를 세 번
                [330, 100, 520, 260], [340, 110, 530, 270]], float)           # 다른 물체를 두 번
toy_s = np.array([0.9, 0.8, 0.7, 0.85, 0.6])
kept = nms(toy, toy_s)
canvas = np.full((360, 600, 3), 255, np.uint8)
before, after = canvas.copy(), canvas.copy()
for b, s in zip(toy.astype(int), toy_s):
    cv2.rectangle(before, tuple(b[:2]), tuple(b[2:]), (0, 0, 255), 2)
for k in kept:
    b = toy[k].astype(int)
    cv2.rectangle(after, tuple(b[:2]), tuple(b[2:]), (0, 160, 0), 3)
    cv2.putText(after, f"{toy_s[k]:.2f}", (b[0], b[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 160, 0), 2)
print("남긴 상자:", kept, "(왼쪽: NMS 전 5개, 오른쪽: NMS 뒤)")
show(np.hstack([before, after]), 900)

### 진짜 출력에 적용 — 거르기 → 모서리 좌표 → 종류별 NMS
- 모델은 상자를 **(중심, 크기)** 로 낸다. IoU 는 **(모서리)** 로 계산하므로 바꾼다: `x1 = cx − w/2`, `x2 = cx + w/2` …
- **종류별 NMS**: 사람 상자가 겹쳐 있는 개 상자를 지우면 안 된다. 종류마다 NMS 를 따로 돌려야 하는데,
  **종류 번호 × 4096 만큼 좌표를 밀어 두면** 종류가 다른 상자끼리는 절대 안 겹치므로 NMS 한 번으로 끝난다 (흔히 쓰는 요령).

In [ ]:
def decode(pred, conf_th=0.25, iou_th=0.45):
    scores_all = pred[:, 4:]
    cls, conf = scores_all.argmax(1), scores_all.max(1)
    m = conf > conf_th                                # ① 점수 거르기
    b, cls, conf = pred[m, :4], cls[m], conf[m]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,   # ② 중심·크기 → 모서리 좌표
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    keep = nms(xyxy + cls[:, None] * 4096.0, conf, iou_th)            # ③ 종류별 NMS (좌표 밀기 요령)
    return xyxy[keep], cls[keep], conf[keep], int(m.sum())

xyxy, cls_k, conf_k, n_pass = decode(pred)
print(f"후보 8400 → 점수 통과 {n_pass} → NMS 뒤 {len(xyxy)}")

In [ ]:
# OpenCV 에도 NMS 가 있다 — 결과가 같은지 확인 (cv2 는 상자를 [x, y, 너비, 높이] 로 받는다)
m = conf > 0.25
b = pred[m, :4]
shifted = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 2], b[:, 3]], 1)
shifted[:, :2] += cls[m][:, None] * 4096.0
idx = cv2.dnn.NMSBoxes(shifted.tolist(), conf[m].tolist(), 0.25, 0.45)
print("직접 만든 NMS:", len(xyxy), "| cv2.dnn.NMSBoxes:", len(idx))

## 8. 원래 사진 좌표로 되돌리고 그리기
letterbox 를 거꾸로: **여백을 빼고 → 배율로 나눈다**. 사진 밖으로 나간 부분은 잘라 낸다(`clip`).

$$x_{\text{원래}} = \frac{x_{640} - \text{왼쪽 여백}}{\text{배율}}, \qquad y_{\text{원래}} = \frac{y_{640} - \text{위 여백}}{\text{배율}}$$

In [ ]:
def unletterbox(xyxy, ratio, pad, shape):
    h, w = shape[:2]
    out = xyxy.copy()
    out[:, [0, 2]] = (out[:, [0, 2]] - pad[0]) / ratio
    out[:, [1, 3]] = (out[:, [1, 3]] - pad[1]) / ratio
    out[:, [0, 2]] = out[:, [0, 2]].clip(0, w)
    out[:, [1, 3]] = out[:, [1, 3]].clip(0, h)
    return out

boxes = unletterbox(xyxy, ratio, pad, frame.shape)
view = frame.copy()
for (x1, y1, x2, y2), c, s in zip(boxes.astype(int), cls_k, conf_k):
    color = tuple(int(v) for v in np.random.default_rng(int(c)).integers(60, 255, 3))   # 종류마다 같은 색
    cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
    cv2.putText(view, f"{NAMES[int(c)]} {s:.2f}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    print(f"{NAMES[int(c)]:14s} {s:.2f}  ({x1}, {y1}) - ({x2}, {y2})")
show(view)

## 9. 어디에 시간이 드나
추론만 빠르다고 끝이 아니다. 전처리(크기 바꾸기 · 색 바꾸기 · 나누기)와 후처리(NMS)는 장치와 상관없이 **CPU** 가 한다.
GPU · NPU 로 추론이 빨라질수록 전·후처리가 차지하는 몫이 커진다.

In [ ]:
def timed(n=20):
    t = {"전처리": 0.0, "추론": 0.0, "후처리": 0.0}
    for _ in range(n):
        t0 = time.perf_counter(); b, r, p = letterbox(frame); x = to_blob(b)
        t1 = time.perf_counter(); o = compiled(x)[compiled.output(0)]
        t2 = time.perf_counter(); decode(o[0].T)
        t3 = time.perf_counter()
        t["전처리"] += t1 - t0; t["추론"] += t2 - t1; t["후처리"] += t3 - t2
    return {k: v / n * 1000 for k, v in t.items()}

for k, v in timed().items():
    print(f"{k:6s} {v:7.2f} ms  {'█' * max(1, round(v))}")

## 10. (덤) 포즈 모델 — 관절 17개
`yolo11m-pose` 의 출력은 `[1, 56, 8400]` 이다: 상자 4 + 사람 점수 1 + 관절 17개 × (x, y, 보임 점수) = 56.
종류가 사람 하나뿐이라 NMS 는 종류별로 나눌 필요가 없다. 관절 좌표도 상자와 똑같이 letterbox 를 되돌린다.

In [ ]:
POSE_DIR = MODELS / "object" / "yolo11m-pose_openvino_model"
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4)]   # COCO 관절 잇기

if not POSE_DIR.exists():
    print("포즈 모델이 없어 건너뜁니다:", POSE_DIR)
else:
    pose = core.compile_model(next(POSE_DIR.glob("*.xml")), DEVICE)
    p = pose(blob)[pose.output(0)][0].T                    # [8400, 56]
    score = p[:, 4]
    m = score > 0.5
    b, score, kpt = p[m, :4], score[m], p[m, 5:].reshape(-1, 17, 3)
    xyxy_p = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    keep = nms(xyxy_p, score)
    view = frame.copy()
    for k in keep:
        pts = kpt[k].copy()
        pts[:, 0] = (pts[:, 0] - pad[0]) / ratio           # 관절도 letterbox 되돌리기
        pts[:, 1] = (pts[:, 1] - pad[1]) / ratio
        for a, c in SKELETON:
            if pts[a, 2] > 0.5 and pts[c, 2] > 0.5:         # 둘 다 보일 때만 잇는다
                cv2.line(view, tuple(pts[a, :2].astype(int)), tuple(pts[c, :2].astype(int)), (255, 160, 0), 3)
        for x, y, v in pts:
            if v > 0.5:
                cv2.circle(view, (int(x), int(y)), 4, (0, 0, 255), -1)
    print("사람", len(keep), "명")
    show(view)

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 아무것도 못 찾음 | `÷255` 를 빠뜨렸거나 BGR 그대로 넣었다 — 4절 확인 |
| 상자가 위아래로 밀림 | 되돌릴 때 여백(`pad`)을 안 뺐다 |
| 한 물체에 상자가 여러 개 | NMS 를 빠뜨렸거나 `iou_th` 가 너무 높다 |
| 사람 위의 가방이 사라짐 | 종류 구분 없이 NMS 를 돌렸다 — 좌표 밀기 요령 |
| `IndexError` (`out[0].T`) | 출력이 `[1, 84, 8400]` 인지 먼저 찍어 본다 |

## 정리
- 전처리: letterbox(배율 · 여백 기억) → RGB → ÷255 → NCHW.
- 출력: 후보 8400개 = 세 크기 격자 칸의 합. 후보마다 중심 · 크기 + 80종 점수.
- 후처리: 점수 거르기 → 모서리 좌표 → (종류별) NMS → letterbox 되돌리기.
- 시간: 추론이 빨라질수록 전·후처리 몫이 커진다.

## 해 볼 것
1. `decode(pred, conf_th=0.5)` · `iou_th=0.7` 로 바꿔 보자. 상자 수가 어떻게 달라지나?
2. letterbox 대신 `cv2.resize(frame, (640, 640))` 로 늘려 넣으면 결과가 어떻게 달라지나? 좌표 되돌리기는 어떻게 바뀌나?
   <details><summary>힌트</summary>

   가로 배율 `640 / w`, 세로 배율 `640 / h` 가 따로다: `x / (640 / w)`, `y / (640 / h)`.
   </details>
3. `DEVICE` 를 `"CPU"` · `"NPU"` 로 바꿔 9절 표를 비교해 보자.
4. 라이브러리 결과와 맞는지 확인해 보자.
   <details><summary>힌트</summary>

   `from ultralytics import YOLO; r = YOLO(str(YOLO_DIR), task="detect")(frame)[0]; print(r.boxes.xyxy, r.boxes.cls, r.boxes.conf)`
   </details>
5. 웹캠 영상에서 "사람" 만 세어 화면에 숫자로 띄워 보자 (`cls_k == 0`).

---
# 🛠 미니 프로젝트 — 사람 수 세기 모니터

**만들 것**: 웹캠 영상에서 **사람(종류 0)만** 세어 화면에 띄우고, 1초마다 시각 · 사람 수를 **CSV** 로 기록한다.
끝나면 가장 많았던 순간과 평균을 알려 준다. (매장 · 교실 혼잡도 측정의 아주 작은 판)

**쓰는 것**: `letterbox` · `to_blob` · `decode` · `unletterbox` (이 노트북 — 라이브러리 없이 만든 것 그대로), `live`

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
import csv
from datetime import datetime

log, last_log = [], [0.0]

def count_people(f):
    b, r, p = letterbox(f)
    out = compiled(to_blob(b))[compiled.output(0)][0].T
    xyxy, c, s, _ = decode(out, conf_th=0.4)
    people = unletterbox(xyxy[c == 0], r, p, f.shape)              # 사람만
    for x1, y1, x2, y2 in people.astype(int):
        cv2.rectangle(f, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(f, f"people: {len(people)}", (10, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)
    if time.time() - last_log[0] >= 1.0:                            # 1초에 한 번 기록
        log.append((datetime.now().strftime("%H:%M:%S"), len(people)))
        last_log[0] = time.time()
    return f

live(count_people, seconds=20)
with open("people_log.csv", "w", newline="", encoding="utf-8") as fp:
    csv.writer(fp).writerows([("time", "people")] + log)
if log:
    counts = [n for _, n in log]
    peak = max(log, key=lambda x: x[1])
    print(f"기록 {len(log)}개 → people_log.csv · 평균 {np.mean(counts):.1f}명 · 최대 {peak[1]}명 ({peak[0]})")

**더 해 보기**
- 사람 대신 다른 종류 세기 (예: 병 39, 컵 41 — `NAMES` 에서 번호 찾기)
- 화면 왼쪽 절반 / 오른쪽 절반의 사람 수를 따로 세기 (상자 가운데 x 로 나누기)
- 사람이 0 명 → 1 명 이상으로 바뀌는 순간만 기록하기 (출입 알림)